In [ ]:
import os
import sys
from pathlib import Path

library_path = os.path.abspath('../../src')
if library_path not in sys.path:
    sys.path.append(library_path)
library_path = Path(library_path)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

DATA_PATH  = library_path.parent / 'data'
PLOTS_PATH = library_path.parent / 'plots'
PLOTS_PATH.mkdir(parents=True, exist_ok=True)

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

## Norm Derivation — Restricted Pipeline

### What this notebook does

This notebook derives population reference norms for seven EQ-5D-5L outcomes by applying
importance-weighted estimation to DAPHNIE 2024, using HSE 2017–2018 as the target population.

### Statistical rationale

The core idea is **importance weighting** (covariate shift correction). The density ratio
$w(x) = P_{\text{target}}(x) / P_{\text{source}}(x)$ transforms an expectation under the
source distribution into one under the target distribution:

$$\mathbb{E}_{\text{target}}[Y] \approx \frac{\sum_i w(x_i)\, y_i}{\sum_i w(x_i)}$$

where the sum is over DAPHNIE 2024 observations. When $w$ is well-estimated, this recovers
the population mean we would observe if DAPHNIE had been drawn from the same frame as HSE.

### Design

- **Density ratio:** re-estimated here from scratch using plain logistic regression on the
  15-variable predictor set — same architecture as notebook 031. Plain LR is the recommended
  scheme (best post-reweighting balance with 3 residual imbalanced variables, ESS ≈ 74.5%).
- **Reference:** HSE 2017–2018 survey-weighted with `wt_int`.
- **DAPHNIE unadjusted:** DAPHNIE 2024 design-weighted with `svy_wt`.
- **DAPHNIE adjusted:** weighted with `svy_wt × w_LR`.
- **Missingness:** complete-case analysis per outcome. ~10% block missing on EQ-5D in HSE
  2017–18 (not MCAR; systematically less healthy — stated limitation for abstract).

In [ ]:
df = pd.read_csv(DATA_PATH / 'wrangled_data.csv', low_memory=False)

hse_1718   = df[df['dataset'].isin(['HSE 2017', 'HSE 2018'])].copy().reset_index(drop=True)
daphnie_24 = df[df['dataset'] == 'DAPHNIE 2024'].copy().reset_index(drop=True)

print(f'HSE 2017–2018: n = {len(hse_1718):,}')
print(f'DAPHNIE 2024:  n = {len(daphnie_24):,}')

In [ ]:
# 15-variable predictor set (notebook 031)
FEATURES = [
    'Sex', 'age7cat',
    'eth2cat',
    'emp_cat_Employed', 'emp_cat_Other (Sick/Home/etc)', 'emp_cat_Retired',
    'emp_cat_Student', 'emp_cat_Unemployed',
    'edu_cat_2',
    'smoke_ecig', 'diabetes',
    'meds_num', 'ill_dis',
    'resp', 'skin',
]
FEATURES = [f for f in FEATURES if f in df.columns]
assert len(FEATURES) == 15, f'Expected 15 features, got {len(FEATURES)}'
print(f'Feature set ({len(FEATURES)}):')
for f in FEATURES:
    print(f'  {f}')

## Step 1: Density Ratio Re-estimation

Plain logistic regression with `class_weight='balanced'` is trained to distinguish
DAPHNIE 2024 (label = 0) from HSE 2017–2018 (label = 1) using the 15 predictors.
Equalising the class prior means the predicted probability odds are directly the
density ratio:

$$w(x) = \frac{\hat{p}(x)}{1 - \hat{p}(x)}$$

No sample-size correction is needed because `class_weight='balanced'` already
absorbs the marginal class imbalance (DAPHNIE:HSE ≈ 1:3.1).

Weights are clipped at the 99th percentile to prevent extreme observations from
dominating the weighted estimator, then renormalised to mean 1.

In [ ]:
def fit_lr_density_ratio(source_df, target_df, features):
    """
    Pool source (label=0) and target (label=1); fit a balanced logistic regression.
    Returns the fitted sklearn Pipeline.
    """
    src    = source_df[features].assign(_label=0)
    tgt    = target_df[features].assign(_label=1)
    pooled = pd.concat([src, tgt], ignore_index=True)
    X, y   = pooled[features], pooled['_label']

    lr = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler()),
        ('clf',     LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)),
    ])
    lr.fit(X, y)
    return lr


def compute_weights(clf, source_X, clip_percentile=99):
    """
    Density ratio weights w(x) = p/(1-p), normalised to mean 1,
    clipped at clip_percentile.
    Returns (weights_array, clip_value, n_clipped, effective_sample_size).
    """
    p_hat = clf.predict_proba(source_X)[:, 1]
    w     = p_hat / (1 - p_hat + 1e-8)
    w     = w / w.mean()

    clip_val  = np.percentile(w, clip_percentile)
    n_clipped = int((w > clip_val).sum())
    w         = np.clip(w, 0, clip_val)
    w         = w / w.mean()

    n_eff = w.sum() ** 2 / (w ** 2).sum()
    return w, clip_val, n_clipped, n_eff

In [ ]:
lr_model = fit_lr_density_ratio(daphnie_24, hse_1718, FEATURES)
w_lr, clip_val, n_clipped, n_eff = compute_weights(lr_model, daphnie_24[FEATURES])

print('LR density ratio weights (99th-pct clip):')
print(f'  Clip value:          {clip_val:.3f}')
print(f'  Clipped observations: {n_clipped} ({100*n_clipped/len(daphnie_24):.1f}%)')
print(f'  ESS:                 {n_eff:.0f} / {len(daphnie_24)} ({100*n_eff/len(daphnie_24):.1f}%)')
print(f'  Weight range:        [{w_lr.min():.3f}, {w_lr.max():.3f}]  mean = {w_lr.mean():.3f}')

## Step 2: Outcome Variables

Seven outcomes are in scope for the abstract analysis:

| Variable | Type | Description |
|---|---|---|
| `MO5L` | ordinal 1–5 | Mobility |
| `SC5L` | ordinal 1–5 | Self-care |
| `UA5L` | ordinal 1–5 | Usual activities |
| `PD5L` | ordinal 1–5 | Pain/discomfort |
| `AD5L` | ordinal 1–5 | Anxiety/depression |
| `EQ_index` | continuous | EQ-5D-5L utility index (UK value set) |
| `LSS_rs` | continuous, 0–100 | Leidelmeijer Severity Score |

`FULLHEALTH` (binary) is derived: 1 iff all five dimensions equal 1. Missingness
propagates — any missing dimension makes FULLHEALTH missing.

All analyses are complete-case per outcome.

In [ ]:
DIMENSIONS  = ['MO5L', 'SC5L', 'UA5L', 'PD5L', 'AD5L']
CONTINUOUS  = ['EQ_index', 'LSS_rs']

# Derive FULLHEALTH: NaN if any dimension is missing
for ds in [daphnie_24, hse_1718]:
    all_obs = ds[DIMENSIONS].notna().all(axis=1)
    ds['FULLHEALTH'] = np.where(
        all_obs,
        (ds[DIMENSIONS] == 1).all(axis=1).astype(float),
        np.nan,
    )

ALL_OUTCOMES = DIMENSIONS + CONTINUOUS + ['FULLHEALTH']

print(f'  {"Outcome":<14}  {"DAPHNIE 2024":>13}  {"HSE 2017–18":>12}')
print(f'  {"-"*42}')
for out in ALL_OUTCOMES:
    m_d = 100 * daphnie_24[out].isna().mean()
    m_h = 100 * hse_1718[out].isna().mean()
    print(f'  {out:<14}  {m_d:>12.1f}%  {m_h:>11.1f}%')

## Step 3: Population Norm Estimation

For each outcome, we estimate the population mean under three conditions:

| Condition | Sample | Weights |
|---|---|---|
| **HSE 2017–18** | HSE 2017–2018 | `wt_int` (official survey weights) |
| **DAPHNIE (unadjusted)** | DAPHNIE 2024 | `svy_wt` (quota-sample design weights) |
| **DAPHNIE (adjusted)** | DAPHNIE 2024 | `svy_wt × w_LR` |

The estimator is the Horvitz–Thompson weighted mean. Standard errors use the sandwich
estimator, which is valid regardless of the weight distribution:

$$\widehat{\mathrm{SE}} = \sqrt{\frac{\sum_i w_i^2 (y_i - \bar{y}_w)^2}{\left(\sum_i w_i\right)^2}}$$

For binary outcomes (FULLHEALTH, % any problem per dimension), this reduces to the
weighted binomial proportion SE. Confidence intervals: $\bar{y}_w \pm 1.96\,\widehat{\mathrm{SE}}$.

In [ ]:
def wmean_se(y, w):
    """
    Weighted mean and sandwich SE.
    y, w: array-like (NaN in y is excluded; only finite positive w are used).
    Returns (mean, se).
    """
    y = np.asarray(y, dtype=float)
    w = np.asarray(w, dtype=float)
    mask = np.isfinite(y) & np.isfinite(w) & (w > 0)
    if mask.sum() == 0:
        return np.nan, np.nan
    y_, w_ = y[mask], w[mask]
    W   = w_.sum()
    mu  = np.dot(w_, y_) / W
    var = np.sum(w_**2 * (y_ - mu)**2) / W**2
    return mu, np.sqrt(var)


# Positional weight arrays (aligned with each reset-indexed dataframe)
HSE_W   = hse_1718['wt_int'].fillna(1.0).values
D24_SWY = daphnie_24['svy_wt'].fillna(1.0).values
D24_ADJ = D24_SWY * w_lr

# (label, dataframe, weight_array) triples used throughout
CONDITIONS = [
    ('HSE 2017–18',          hse_1718,   HSE_W),
    ('DAPHNIE (unadjusted)', daphnie_24, D24_SWY),
    ('DAPHNIE (adjusted)',   daphnie_24, D24_ADJ),
]

COND_COLORS = {
    'HSE 2017–18':          '#2ca02c',
    'DAPHNIE (unadjusted)': '#d62728',
    'DAPHNIE (adjusted)':   '#1f77b4',
}

In [ ]:
# Overall norms table: mean (95% CI) for each outcome under each condition
rows = []
for out in ALL_OUTCOMES:
    for cond_label, ds, wts in CONDITIONS:
        mu, se = wmean_se(ds[out].values, wts)
        n_obs  = int(np.isfinite(ds[out].values).sum())
        rows.append({
            'Outcome':   out,
            'Condition': cond_label,
            'n':         n_obs,
            'Mean':      mu,
            'SE':        se,
            'CI_lo':     mu - 1.96 * se,
            'CI_hi':     mu + 1.96 * se,
        })

norms_long = pd.DataFrame(rows)

# Wide format for display
norms_wide = (
    norms_long
    .pivot(index='Outcome', columns='Condition', values=['n', 'Mean', 'SE'])
    .reindex(ALL_OUTCOMES)
)
print('Population norms — weighted means and SEs:')
display(norms_wide.round(4))

In [ ]:
# % reporting any problem (dimension level >= 2) for the five EQ-5D dimensions
anyproblem_rows = []
for dim in DIMENSIONS:
    for cond_label, ds, wts in CONDITIONS:
        y_bin          = ds[dim].copy().values.astype(float)
        y_bin          = np.where(np.isnan(y_bin), np.nan, (y_bin >= 2).astype(float))
        mu, se         = wmean_se(y_bin, wts)
        anyproblem_rows.append({
            'Dimension': dim,
            'Condition': cond_label,
            'pct_any':   100 * mu,
            'pct_se':    100 * se,
        })

ap_df = pd.DataFrame(anyproblem_rows)
ap_wide = ap_df.pivot(index='Dimension', columns='Condition', values=['pct_any', 'pct_se'])
print('% reporting any problem (level ≥ 2):')
display(ap_wide.reindex(DIMENSIONS).round(2))

In [ ]:
# Weighted % at each level (1–5) for each EQ-5D dimension
level_rows = []
for dim in DIMENSIONS:
    for cond_label, ds, wts in CONDITIONS:
        for lvl in range(1, 6):
            y_bin      = np.where(np.isnan(ds[dim].values), np.nan,
                                  (ds[dim].values == lvl).astype(float))
            mu, se     = wmean_se(y_bin, wts)
            level_rows.append({
                'Dimension': dim, 'Level': lvl,
                'Condition': cond_label,
                'pct':       100 * mu,
            })

level_df = pd.DataFrame(level_rows)
level_wide = (
    level_df
    .pivot(index=['Dimension', 'Level'], columns='Condition', values='pct')
    .reindex(pd.MultiIndex.from_product([DIMENSIONS, range(1, 6)],
                                        names=['Dimension', 'Level']))
)
print('Weighted % at each level (1–5):')
display(level_wide.round(1))

In [ ]:
# Dimension profile plot: grouped bar per level for each dimension
DIM_LABELS = {
    'MO5L': 'Mobility',
    'SC5L': 'Self-care',
    'UA5L': 'Usual activities',
    'PD5L': 'Pain/discomfort',
    'AD5L': 'Anxiety/depression',
}

x       = np.arange(1, 6)
width   = 0.25
offsets = [-width, 0, width]

fig, axes = plt.subplots(1, 5, figsize=(18, 4.5), sharey=False)
for ax, dim in zip(axes, DIMENSIONS):
    for offset, (cond_label, ds, wts) in zip(offsets, CONDITIONS):
        pcts = [
            100 * wmean_se(
                np.where(np.isnan(ds[dim].values), np.nan, (ds[dim].values == lvl).astype(float)),
                wts
            )[0]
            for lvl in range(1, 6)
        ]
        ax.bar(x + offset, pcts, width=width, label=cond_label,
               color=COND_COLORS[cond_label], alpha=0.85)

    ax.set_title(DIM_LABELS[dim], fontsize=9)
    ax.set_xlabel('Level')
    ax.set_xticks(x)
    ax.set_ylabel('% respondents')

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper right', fontsize=8, framealpha=0.9)
fig.suptitle('EQ-5D-5L dimension profiles: HSE 2017–18 vs DAPHNIE 2024',
             fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig(PLOTS_PATH / 'eq5d_dimension_profiles.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Weighted KDE for EQ_index and LSS_rs
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, out in zip(axes, CONTINUOUS):
    for cond_label, ds, wts in CONDITIONS:
        y    = ds[out].values
        w    = wts.copy()
        mask = np.isfinite(y) & np.isfinite(w) & (w > 0)
        y_,  w_ = y[mask], w[mask]
        w_       = w_ / w_.sum()         # normalise to 1 for KDE
        sns.kdeplot(
            x=y_, weights=w_, ax=ax,
            label=cond_label,
            color=COND_COLORS[cond_label],
            linewidth=1.8,
        )
    mu_h = wmean_se(hse_1718[out].values, HSE_W)[0]
    ax.axvline(mu_h, color=COND_COLORS['HSE 2017–18'],
               linewidth=1.0, linestyle='--', alpha=0.7)
    ax.set_title(out)
    ax.set_xlabel(out)
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)

fig.suptitle('Outcome distributions — weighted KDE', fontsize=11, y=1.01)
plt.tight_layout()
plt.savefig(PLOTS_PATH / 'continuous_outcome_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 4: Subgroup Norms by Sex × Age Group

EuroQoL population norms are conventionally reported stratified by sex and age group.
We use the `age7cat` variable (seven 10-year bands) and `Sex` (binary).

Weights for each subgroup are the same as overall — we restrict to the subgroup
and apply the pre-computed `svy_wt × w_LR` without renormalising within the
subgroup, preserving cross-subgroup comparability.

In [ ]:
# Inspect coding before applying labels
print('age7cat distribution (DAPHNIE 2024):')
print(daphnie_24['age7cat'].value_counts().sort_index().to_string())
print()
print('age7cat distribution (HSE 2017–18):')
print(hse_1718['age7cat'].value_counts().sort_index().to_string())
print()
print('Sex distribution (DAPHNIE 2024):')
print(daphnie_24['Sex'].value_counts().sort_index().to_string())
print()
print('Sex distribution (HSE 2017–18):')
print(hse_1718['Sex'].value_counts().sort_index().to_string())

In [ ]:
# Adjust these mappings if the coding printout above differs from the labels here
AGE_LABELS = {1: '18–24', 2: '25–34', 3: '35–44', 4: '45–54',
              5: '55–64', 6: '65–74', 7: '75+'}
SEX_LABELS = {1: 'Male', 2: 'Female'}


def subgroup_norms(outcome, conditions, age_labels, sex_labels):
    """
    Compute weighted mean (and SE) for `outcome` within each Sex x age7cat cell.
    Returns a long-format DataFrame.
    """
    all_ages = sorted({int(v) for ds, _ in [(c[1], c[2]) for c in conditions]
                       for v in ds['age7cat'].dropna().unique()})
    all_sex  = sorted({int(v) for ds, _ in [(c[1], c[2]) for c in conditions]
                       for v in ds['Sex'].dropna().unique()})

    rows = []
    for sex_val in all_sex:
        for age_val in all_ages:
            for cond_label, ds, wts_arr in conditions:
                mask   = ((ds['Sex'] == sex_val) & (ds['age7cat'] == age_val)).values
                y_sub  = ds[outcome].values[mask]
                w_sub  = wts_arr[mask]
                mu, se = wmean_se(y_sub, w_sub)
                rows.append({
                    'Sex':       sex_labels.get(sex_val,  str(sex_val)),
                    'Age group': age_labels.get(age_val,  str(age_val)),
                    'Condition': cond_label,
                    'n':         int(mask.sum()),
                    'mean':      mu,
                    'SE':        se,
                })
    return pd.DataFrame(rows)


# EQ_index subgroup norms
sg_eq = subgroup_norms('EQ_index', CONDITIONS, AGE_LABELS, SEX_LABELS)
sg_eq_wide = (
    sg_eq
    .pivot(index=['Sex', 'Age group'], columns='Condition', values=['n', 'mean', 'SE'])
)
print('EQ-5D-5L utility index — weighted norms by sex and age group:')
display(sg_eq_wide.round(4))

In [ ]:
# Subgroup norms for all abstract outcomes
sg_all = {}
for out in ALL_OUTCOMES:
    sg_all[out] = subgroup_norms(out, CONDITIONS, AGE_LABELS, SEX_LABELS)

# Combined table: mean only, one outcome per column-group, sex x age as index
means_by_cond = {}
for out in ALL_OUTCOMES:
    tmp = sg_all[out].pivot(index=['Sex', 'Age group'], columns='Condition', values='mean')
    tmp.columns = pd.MultiIndex.from_product([[out], tmp.columns])
    means_by_cond[out] = tmp

sg_combined = pd.concat(means_by_cond.values(), axis=1)
print('Subgroup norms — weighted means by Sex × Age group (all abstract outcomes):')
display(sg_combined.round(4))

In [ ]:
# EQ_index subgroup norm plot: adjusted DAPHNIE vs HSE reference
age_order = [AGE_LABELS[k] for k in sorted(AGE_LABELS)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, sex_lbl in zip(axes, ['Male', 'Female']):
    for cond_label, ds, wts_arr in CONDITIONS:
        sub = sg_eq[(sg_eq['Sex'] == sex_lbl) & (sg_eq['Condition'] == cond_label)]
        sub = sub.set_index('Age group').reindex(age_order)
        ax.plot(
            age_order, sub['mean'],
            marker='o', linewidth=1.5, markersize=5,
            label=cond_label, color=COND_COLORS[cond_label],
        )
        ax.fill_between(
            age_order,
            sub['mean'] - 1.96 * sub['SE'],
            sub['mean'] + 1.96 * sub['SE'],
            color=COND_COLORS[cond_label], alpha=0.10,
        )

    ax.set_title(sex_lbl)
    ax.set_xlabel('Age group')
    ax.set_ylabel('EQ-5D-5L utility index')
    ax.tick_params(axis='x', rotation=45)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=3, fontsize=8, framealpha=0.9)
fig.suptitle('EQ-5D-5L utility index by sex and age group\n'
             'HSE 2017–18 vs DAPHNIE 2024 (unadjusted and adjusted)',
             fontsize=11)
plt.tight_layout(rect=[0, 0.08, 1, 1])
plt.savefig(PLOTS_PATH / 'eq_index_subgroup_norms.png', dpi=150, bbox_inches='tight')
plt.show()